# Notebook for running classification experiments

- [Import and initial setup](#import)
- [Experiment setup](#ex)
- [Read and clean data](#read)
- [Set up embedder and DB collection](#embed)
- [Run experiment and queries to LLM](#run)
- [Write and save results to disk](#save)

<a id="import"></a>
## Import and initial setup

In [ ]:
import pandas as pd
import os
import getpass
import requests
import re
import time
import json
import s3fs

from klass import get_classification

import chromadb
from sentence_transformers import SentenceTransformer
from sklearn.metrics import f1_score, accuracy_score

from utils.create_db import download_chroma_from_s3
from utils.formatting import reduce_test, write_results, get_sn
from utils.rag_setup import query_db, format_few_shot_examples, 
from utils.prompt_setup import intro_prompt, build_prompt_standard, build_prompt_zero, build_prompt_descriptions, build_prompt_obs, build_prompt_fasttext, format_candidates

from config import embeddings_model, chromadb_path, collection_name_class, collection_name_obs, train_path, test_path, llm_model_onyxia

In [ ]:
# Set LLM model token
if os.getenv("LLM_TOKEN") is None: 
    os.environ["LLM_TOKEN"] = getpass.getpass("Model token: ")

<a id="ex"></a>
## Experiment setup
Here you can specify want typ of setup to use for the experiment. Choose between:

- 'zero': No RAG or extra descriptions in the prompt.
- 'standard': No RAG but a full description of the NACE classes is provided.
- 'observed': RAG using observations in the DB with 5 most similar being collected into the prompt.
- 'descriptions': RAG using NACE class descriptions in the DB with 5 most similar collected into the prompt.
- 'fasttext': Augmented prompt using descriptions of the top 5 NACE classes from the Fasttext model.

In [ ]:
rag_type = "fasttext" 

Here you can specify the number of observations in the test data. Due to computation time, we have choosen to run experiments on a samll subset of 100

In [ ]:
max_nr = 100

<a id="read"></a>
## Read and clean data

In [ ]:
# Read in data and reduce
test = pd.read_parquet(test_path)
test = test.reset_index(drop=True)
test = reduce_test(test)
print(test.columns)

# Test data descriptions
tab = test.groupby('nace_21_code').orgnr.count()
print(f'Test data includes this number of NACE groups {len(tab)}')
print(f'Min obs in NACE groups {min(tab)}')
print(f'Max obs in NACE groups {max(tab)}')

<a id="embed"></a>
## Set up embedder and DB collection

In [ ]:
embedder = SentenceTransformer(embeddings_model)

In [ ]:
# Set up connection to SSPcloud
S3_BUCKET = "projet-aiml4os-wp10/Cluster2/Norway-RAG-data"
S3_PREFIX = "chroma_storage"
S3_ENDPOINT_URL = "https://" + os.environ["AWS_S3_ENDPOINT"]
fs = s3fs.S3FileSystem(client_kwargs={'endpoint_url': S3_ENDPOINT_URL})

In [ ]:
# Download chromadb if it exists
if not os.path.exists(os.path.join(chromadb_path, "chroma.sqlite3")):
    download_chroma_from_s3(S3_BUCKET, S3_PREFIX, chromadb_path, fs)
else:
    print("Chroma already exists → skipping download")

In [ ]:
# Set up collection
client = chromadb.PersistentClient(path=chromadb_path)
if rag_type == "standard":
    collection_db = collection_name_class 
elif rag_type == "observed":
    collection_db = collection_name_obs
else: 
    collection_db = collection_name_class # Set this but not used in zero or fasttext
collection = client.get_or_create_collection(
    name=collection_db,
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)
print(f'Number of chunks in collection: {collection.count()}')

In [ ]:
# Setup specifics based on rag type
if rag_type == "fasttext":
    # Get SN
    sn = get_sn()
    print("NACE codes collected")

    # Get top 5 hits from fasttext
    top5_path = 'src/temp/top5_predictions_small.json' # Just using relative path here but a bit unreliable!
    with open(top5_path) as f:
        top5 = json.load(f)
    print("Top 5 fastext codes collected")  

if rag_type == "descriptions":
    sn = get_sn()
    
    train = pd.read_parquet(train_path)
    all_nace = train.nace_21_code.unique()
    all_nace = all_nace[all_nace != "00.000"]
    sn_descriptions = format_candidates(all_nace, sn, notes = False)
    

<a id="run"></a>
## Run experiment and queries to LLM

In [ ]:
# Set up access to model
headers = {
    "Authorization": f'Bearer {os.getenv("LLM_TOKEN")}',
    "Content-Type": "application/json"
}

classification_results = {}
t_start = time.time()

for i in range(max_nr):
    t = time.time()
    company_temp = test.iloc[i,:]
    if rag_type in ("standard", "observed"):
        hits_df = query_db(company_temp["company_activity"], collection, embedder, rag_type)
        if rag_type == 'standard':
            prompt = build_prompt_standard(company_temp["company_name"], 
                              company_temp["company_activity"],
                              company_temp["company_purpose"],  
                              hits_df)
            hits_codes = [h["code"] for h in hits_df]
        else:
            prompt = build_prompt_obs(company_temp["company_name"], 
                          company_temp["company_activity"],
                          company_temp["company_purpose"], 
                          hits_df)
            hits_codes = [h["nace_21_code"] for h in hits_df]
    elif rag_type == "fasttext":
        hits = top5[i]
        prompt = build_prompt_fasttext(company_temp["company_name"], 
                                  company_temp["company_activity"],
                                  company_temp["company_purpose"],   
                                  hits, 
                                  sn)
        hits_codes = hits
        
    elif rag_type == "zero":
        prompt = build_prompt_zero(
                              company_temp["company_name"], 
                              company_temp["company_activity"],
                              company_temp["company_purpose"]
        )
        hits_codes = ""
    elif rag_type == "descriptions":
        prompt = build_prompt_descriptions(
                            company_temp["company_name"], 
                              company_temp["company_activity"],
                              company_temp["company_purpose"],
                    sn_descriptions,
        )
        hits_codes = ""
        
    else:
        raise ValueError("Invalid rag_type input")
    
    payload = {
        "model": llm_model["name"],
        "messages": [
            {"role": "user", "content": prompt}
        ],
    }

    response = requests.post(llm_model["url"], headers = headers, json=payload)
    response.raise_for_status()
    raw_text = response.json()["choices"][0]["message"]["content"]
    answer_only = re.sub(r"<think>.*?</think>", "", raw_text, flags=re.DOTALL).strip()
    
    classification_results[i] = {"observed_code": company_temp.nace_21_code,
                                "retrieved_codes": hits_codes,
                                 "predicted": answer_only,
                                }
    print(f"Classifying company nr {i} of {max_nr} took {round(time.time() - t, 2)} seconds")

# Summary results
total_time = time.time() - t_start

<a id="save"></a>
## Write and save results to disk

In [ ]:
write_results(classification_results, rag_type = rag_type, max_nr = max_nr, total_time = total_time, llm_model = llm_model) 